# Prediksi Tingkat Kecemasan Berdasarkan Faktor Gaya Hidup dan Kondisi Klinis

**Mata Kuliah:** Pembelajaran Mesin  
**Metode:** Random Forest · XGBoost · Multi-Layer Perceptron (MLP)  
**Dataset:** Enhanced Anxiety Dataset (11.000 baris × 19 kolom)

---

## Daftar Isi
1. [Setup & Instalasi](#1-setup--instalasi)
2. [Load & Inspeksi Data](#2-load--inspeksi-data)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda)
4. [Preprocessing](#4-preprocessing)
5. [Split Data](#5-split-data)
6. [Modeling](#6-modeling)
   - 6.1 Random Forest
   - 6.2 XGBoost
   - 6.3 Multi-Layer Perceptron (MLP)
7. [Evaluasi & Perbandingan Model](#7-evaluasi--perbandingan-model)
8. [Feature Importance](#8-feature-importance)
9. [Simpan Model Terbaik](#9-simpan-model-terbaik)


## 1. Setup & Instalasi

In [ ]:
# Install library yang diperlukan (jalankan sekali)
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn imbalanced-learn -q


In [ ]:
# ── Import semua library ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# XGBoost
from xgboost import XGBClassifier

# Imbalanced-learn (SMOTE untuk menangani class imbalance)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Joblib (simpan model)
import joblib

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("✅ Semua library berhasil di-import.")
print(f"   Pandas  : {pd.__version__}")
print(f"   NumPy   : {np.__version__}")
print(f"   Sklearn : {__import__('sklearn').__version__}")
print(f"   XGBoost : {__import__('xgboost').__version__}")


## 2. Load & Inspeksi Data

In [ ]:
# ── Mount Google Drive (opsional) ─────────────────────────────────────
# Jika dataset disimpan di Google Drive, uncomment blok di bawah:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/dataset/enhanced_anxiety_dataset.xlsx'

# ── Upload manual (default) ───────────────────────────────────────────
# Jalankan cell ini lalu upload file enhanced_anxiety_dataset.xlsx
from google.colab import files
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print(f"📂 File diupload: {DATA_PATH}")


In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────
df = pd.read_excel(DATA_PATH)

print("=" * 55)
print(f"  Ukuran dataset : {df.shape[0]:,} baris × {df.shape[1]} kolom")
print("=" * 55)
df.head(5)


In [ ]:
# ── Tipe data & missing values ────────────────────────────────────────
info_df = pd.DataFrame({
    'Dtype'        : df.dtypes,
    'Non-Null'     : df.notnull().sum(),
    'Missing'      : df.isnull().sum(),
    'Missing (%)'  : (df.isnull().mean() * 100).round(2),
    'Unique Values': df.nunique()
})
print(info_df.to_string())


In [ ]:
# ── Statistik deskriptif fitur numerik ───────────────────────────────
df.describe().round(2)


## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribusi Target (Anxiety Level)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram distribusi raw
ax1 = axes[0]
counts = df['Anxiety Level (1-10)'].value_counts().sort_index()
colors = ['#3266ad' if v <= 4 else '#e08c3a' if v <= 7 else '#c0392b'
          for v in counts.index]
ax1.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.5)
ax1.set_title('Distribusi Anxiety Level (Raw)', fontweight='bold')
ax1.set_xlabel('Anxiety Level (1–10)')
ax1.set_ylabel('Jumlah')
ax1.set_xticks(range(1, 11))
for i, (x, y) in enumerate(zip(counts.index, counts.values)):
    ax1.text(x, y + 20, str(y), ha='center', fontsize=8)

# Pie chart kelas (3-class)
df['Anxiety_Class'] = pd.cut(
    df['Anxiety Level (1-10)'],
    bins=[0, 4, 7, 10],
    labels=['Low (1–4)', 'Medium (5–7)', 'High (8–10)']
)
class_counts = df['Anxiety_Class'].value_counts()
ax2 = axes[1]
ax2.pie(
    class_counts.values,
    labels=class_counts.index,
    colors=['#3266ad', '#e08c3a', '#c0392b'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
ax2.set_title('Proporsi Kelas (3-class)', fontweight='bold')

plt.suptitle('Distribusi Target: Anxiety Level', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nDistribusi kelas:")
for cls, cnt in class_counts.items():
    print(f"  {cls:18s}: {cnt:,} ({cnt/len(df)*100:.1f}%)")


### 3.2 Korelasi Fitur Numerik terhadap Anxiety

In [ ]:
num_cols = [
    'Age', 'Sleep Hours', 'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)', 'Alcohol Consumption (drinks/week)',
    'Stress Level (1-10)', 'Heart Rate (bpm)',
    'Breathing Rate (breaths/min)', 'Sweating Level (1-5)',
    'Therapy Sessions (per month)', 'Diet Quality (1-10)'
]

corr = df[num_cols + ['Anxiety Level (1-10)']].corr()['Anxiety Level (1-10)'].drop('Anxiety Level (1-10)').sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#3266ad' if v < 0 else '#c0392b' for v in corr.values]
bars = ax.barh(corr.index, corr.values, color=bar_colors, edgecolor='white', height=0.65)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Korelasi Pearson: Fitur Numerik vs Anxiety Level', fontweight='bold')
ax.set_xlabel('Korelasi (r)')
for bar, val in zip(bars, corr.values):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()


### 3.3 Pengaruh Faktor Gaya Hidup

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Stress level vs mean anxiety
stress_anx = df.groupby('Stress Level (1-10)')['Anxiety Level (1-10)'].mean()
axes[0].plot(stress_anx.index, stress_anx.values, 'o-', color='#c0392b',
             linewidth=2, markersize=7)
axes[0].fill_between(stress_anx.index, stress_anx.values, alpha=0.15, color='#c0392b')
axes[0].set_title('Stress Level vs Rata-rata Anxiety', fontweight='bold')
axes[0].set_xlabel('Stress Level (1–10)')
axes[0].set_ylabel('Mean Anxiety Level')
axes[0].grid(axis='y', alpha=0.3)

# Sleep hours vs anxiety
bins_sleep = [0, 5, 6, 7, 8, 15]
labels_sleep = ['< 5 jam', '5–6 jam', '6–7 jam', '7–8 jam', '> 8 jam']
df['SleepBin'] = pd.cut(df['Sleep Hours'], bins=bins_sleep, labels=labels_sleep)
sleep_anx = df.groupby('SleepBin', observed=True)['Anxiety Level (1-10)'].mean()
bar_c = ['#c0392b', '#e08c3a', '#e08c3a', '#3266ad', '#3266ad']
axes[1].bar(sleep_anx.index, sleep_anx.values, color=bar_c, edgecolor='white')
axes[1].set_title('Durasi Tidur vs Rata-rata Anxiety', fontweight='bold')
axes[1].set_xlabel('Kategori Jam Tidur')
axes[1].set_ylabel('Mean Anxiety Level')
for i, v in enumerate(sleep_anx.values):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Pengaruh Faktor Gaya Hidup terhadap Anxiety', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 3.4 Faktor Biner dan Pekerjaan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Faktor biner
binary_cols = ['Smoking', 'Dizziness', 'Medication',
               'Family History of Anxiety', 'Recent Major Life Event']
x = np.arange(len(binary_cols))
width = 0.35
means_no  = [df[df[c] == 'No']['Anxiety Level (1-10)'].mean() for c in binary_cols]
means_yes = [df[df[c] == 'Yes']['Anxiety Level (1-10)'].mean() for c in binary_cols]
bars_no  = axes[0].bar(x - width/2, means_no,  width, label='No',  color='#3266ad', edgecolor='white')
bars_yes = axes[0].bar(x + width/2, means_yes, width, label='Yes', color='#c0392b', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Smoking','Dizziness','Medication','Family
History','Life
Event'], fontsize=9)
axes[0].set_ylim(3.0, 4.8)
axes[0].set_ylabel('Mean Anxiety Level')
axes[0].set_title('Faktor Biner vs Anxiety Level', fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for bar in [*bars_no, *bars_yes]:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7.5)

# Rata-rata anxiety per pekerjaan
occ_anx = df.groupby('Occupation')['Anxiety Level (1-10)'].mean().sort_values(ascending=False)
axes[1].barh(occ_anx.index, occ_anx.values, color='#6b74d4', edgecolor='white')
axes[1].set_xlim(3.4, 4.4)
axes[1].set_xlabel('Mean Anxiety Level')
axes[1].set_title('Rata-rata Anxiety per Pekerjaan', fontweight='bold')
for i, v in enumerate(occ_anx.values):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8.5)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Faktor Risiko & Profil Pekerjaan', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Preprocessing

### 4.1 Definisi Kolom & Label Target

In [ ]:
# ── Hapus kolom bantu yang dibuat saat EDA ────────────────────────────
cols_to_drop = ['Anxiety_Class', 'SleepBin']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).copy()

# ── Definisi kelompok kolom ───────────────────────────────────────────
NUMERIC_COLS = [
    'Age', 'Sleep Hours', 'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)', 'Alcohol Consumption (drinks/week)',
    'Stress Level (1-10)', 'Heart Rate (bpm)',
    'Breathing Rate (breaths/min)', 'Sweating Level (1-5)',
    'Therapy Sessions (per month)', 'Diet Quality (1-10)'
]

BINARY_COLS = [
    'Smoking', 'Family History of Anxiety',
    'Dizziness', 'Medication', 'Recent Major Life Event'
]

MULTI_COLS = ['Gender', 'Occupation']

TARGET_RAW = 'Anxiety Level (1-10)'

# ── Buat label kelas (3-class classification) ─────────────────────────
# Low   : Anxiety 1–4  → 0
# Medium: Anxiety 5–7  → 1
# High  : Anxiety 8–10 → 2
def make_label(x):
    if x <= 4:   return 0  # Low
    elif x <= 7: return 1  # Medium
    else:        return 2  # High

df_clean['target'] = df_clean[TARGET_RAW].apply(make_label)

label_map = {0: 'Low (1–4)', 1: 'Medium (5–7)', 2: 'High (8–10)'}
print("✅ Label target dibuat:")
print(df_clean['target'].value_counts().rename(label_map).to_string())


### 4.2 Encoding Fitur Kategorik

In [ ]:
df_enc = df_clean.copy()

# ── Binary encoding (Yes/No → 1/0) ───────────────────────────────────
for col in BINARY_COLS:
    df_enc[col] = df_enc[col].map({'Yes': 1, 'No': 0})

# ── One-Hot Encoding untuk Gender & Occupation ────────────────────────
df_enc = pd.get_dummies(df_enc, columns=MULTI_COLS, drop_first=False)

# ── Hapus kolom target raw ────────────────────────────────────────────
df_enc.drop(columns=[TARGET_RAW], inplace=True)

# ── Ringkasan ─────────────────────────────────────────────────────────
X = df_enc.drop(columns=['target'])
y = df_enc['target']

print(f"✅ Preprocessing selesai.")
print(f"   Shape X : {X.shape}  (baris × fitur)")
print(f"   Shape y : {y.shape}")
print(f"\nDaftar fitur setelah encoding ({len(X.columns)} total):")
print(X.columns.tolist())


### 4.3 Normalisasi Fitur Numerik

In [ ]:
# StandardScaler hanya diterapkan pada fitur numerik
# (Binary dan dummy columns tidak perlu discale)
scaler = StandardScaler()

# Ambil nama kolom numerik yang masih ada di X
num_in_X = [c for c in NUMERIC_COLS if c in X.columns]

X_scaled = X.copy()
X_scaled[num_in_X] = scaler.fit_transform(X[num_in_X])

print("✅ Normalisasi StandardScaler diterapkan pada fitur numerik.")
print(f"   Kolom yang dinormalisasi: {num_in_X}")
print(f"\nContoh statistik setelah scaling (mean ≈ 0, std ≈ 1):")
print(pd.DataFrame(X_scaled[num_in_X]).describe().round(3).loc[['mean','std']].to_string())


### 4.4 Menangani Class Imbalance dengan SMOTE

In [ ]:
# Cek distribusi sebelum SMOTE
print("Distribusi kelas SEBELUM SMOTE:")
for k, v in y.value_counts().sort_index().items():
    print(f"  {label_map[k]:20s}: {v:,}")

# Terapkan SMOTE
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print("\nDistribusi kelas SETELAH SMOTE:")
unique, counts = np.unique(y_resampled, return_counts=True)
for k, v in zip(unique, counts):
    print(f"  {label_map[k]:20s}: {v:,}")

print(f"\nTotal sampel setelah SMOTE: {len(X_resampled):,}")


## 5. Split Data

In [ ]:
# 80% train — 20% test, stratified berdasarkan kelas
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.20,
    random_state=SEED,
    stratify=y_resampled
)

print(f"✅ Data berhasil di-split:")
print(f"   Train : {X_train.shape[0]:,} sampel ({X_train.shape[0]/len(X_resampled)*100:.0f}%)")
print(f"   Test  : {X_test.shape[0]:,} sampel ({X_test.shape[0]/len(X_resampled)*100:.0f}%)")
print(f"   Fitur : {X_train.shape[1]}")

# Verifikasi distribusi kelas di train & test
print("\nDistribusi kelas di train:")
for k in sorted(np.unique(y_train)):
    cnt = np.sum(y_train == k)
    print(f"  {label_map[k]:20s}: {cnt:,} ({cnt/len(y_train)*100:.1f}%)")

print("\nDistribusi kelas di test:")
for k in sorted(np.unique(y_test)):
    cnt = np.sum(y_test == k)
    print(f"  {label_map[k]:20s}: {cnt:,} ({cnt/len(y_test)*100:.1f}%)")


## 6. Modeling

### 6.1 Random Forest Classifier

Random Forest adalah ensemble learning yang membangun banyak decision tree secara paralel
menggunakan teknik **bagging** (Bootstrap Aggregating). Prediksi akhir ditentukan melalui
voting mayoritas dari seluruh pohon. RF robust terhadap overfitting dan mampu menangani
fitur campuran numerik–kategorik.


In [ ]:
print("🌲 Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators   = 200,
    max_depth      = None,       # pohon tumbuh penuh
    min_samples_split = 5,
    min_samples_leaf  = 2,
    max_features   = 'sqrt',
    class_weight   = 'balanced',
    random_state   = SEED,
    n_jobs         = -1
)
rf_model.fit(X_train, y_train)
print("✅ Random Forest selesai dilatih.")

# Prediksi
rf_pred  = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)

# Metrik
rf_acc  = accuracy_score(y_test, rf_pred)
rf_f1   = f1_score(y_test, rf_pred, average='weighted')
rf_auc  = roc_auc_score(y_test, rf_proba, multi_class='ovr', average='weighted')

print(f"\n📊 Hasil Random Forest:")
print(f"   Accuracy  : {rf_acc:.4f} ({rf_acc*100:.2f}%)")
print(f"   F1-Score  : {rf_f1:.4f}")
print(f"   ROC-AUC   : {rf_auc:.4f}")


In [ ]:
# Classification report lengkap
print("📋 Classification Report — Random Forest:")
print(classification_report(
    y_test, rf_pred,
    target_names=[label_map[i] for i in sorted(label_map)],
    digits=4
))


### 6.2 XGBoost Classifier

XGBoost (eXtreme Gradient Boosting) adalah algoritma boosting yang membangun tree secara
**sekuensial**, di mana setiap pohon baru memperbaiki kesalahan pohon sebelumnya dengan
meminimalkan fungsi loss menggunakan gradient descent. XGBoost dikenal efisien dan sering
unggul pada data tabular terstruktur.


In [ ]:
print("⚡ Training XGBoost...")
xgb_model = XGBClassifier(
    n_estimators    = 300,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    reg_alpha       = 0.1,
    reg_lambda      = 1.0,
    use_label_encoder = False,
    eval_metric     = 'mlogloss',
    random_state    = SEED,
    n_jobs          = -1
)
xgb_model.fit(
    X_train, y_train,
    eval_set  = [(X_test, y_test)],
    verbose   = False
)
print("✅ XGBoost selesai dilatih.")

# Prediksi
xgb_pred  = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)

# Metrik
xgb_acc  = accuracy_score(y_test, xgb_pred)
xgb_f1   = f1_score(y_test, xgb_pred, average='weighted')
xgb_auc  = roc_auc_score(y_test, xgb_proba, multi_class='ovr', average='weighted')

print(f"\n📊 Hasil XGBoost:")
print(f"   Accuracy  : {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")
print(f"   F1-Score  : {xgb_f1:.4f}")
print(f"   ROC-AUC   : {xgb_auc:.4f}")


In [ ]:
print("📋 Classification Report — XGBoost:")
print(classification_report(
    y_test, xgb_pred,
    target_names=[label_map[i] for i in sorted(label_map)],
    digits=4
))


### 6.3 Multi-Layer Perceptron (MLP)

MLP adalah jenis jaringan saraf tiruan feedforward yang terdiri dari satu atau lebih
**hidden layer**. Setiap neuron menggunakan fungsi aktivasi non-linear (ReLU) untuk
menangkap hubungan non-linear antar fitur. Bobot dioptimasi menggunakan algoritma
backpropagation dengan optimizer Adam.


In [ ]:
print("🧠 Training MLP (Neural Network)...")
mlp_model = MLPClassifier(
    hidden_layer_sizes = (256, 128, 64),
    activation         = 'relu',
    solver             = 'adam',
    alpha              = 0.001,          # L2 regularization
    batch_size         = 256,
    learning_rate      = 'adaptive',
    learning_rate_init = 0.001,
    max_iter           = 300,
    early_stopping     = True,
    validation_fraction= 0.10,
    n_iter_no_change   = 15,
    random_state       = SEED
)
mlp_model.fit(X_train, y_train)
print(f"✅ MLP selesai dilatih ({mlp_model.n_iter_} iterasi).")

# Prediksi
mlp_pred  = mlp_model.predict(X_test)
mlp_proba = mlp_model.predict_proba(X_test)

# Metrik
mlp_acc  = accuracy_score(y_test, mlp_pred)
mlp_f1   = f1_score(y_test, mlp_pred, average='weighted')
mlp_auc  = roc_auc_score(y_test, mlp_proba, multi_class='ovr', average='weighted')

print(f"\n📊 Hasil MLP:")
print(f"   Accuracy  : {mlp_acc:.4f} ({mlp_acc*100:.2f}%)")
print(f"   F1-Score  : {mlp_f1:.4f}")
print(f"   ROC-AUC   : {mlp_auc:.4f}")


In [ ]:
print("📋 Classification Report — MLP:")
print(classification_report(
    y_test, mlp_pred,
    target_names=[label_map[i] for i in sorted(label_map)],
    digits=4
))


## 7. Evaluasi & Perbandingan Model

### 7.1 Tabel Perbandingan Performa

In [ ]:
results = pd.DataFrame({
    'Model'    : ['Random Forest', 'XGBoost', 'MLP (Neural Network)'],
    'Accuracy' : [rf_acc,  xgb_acc,  mlp_acc],
    'F1-Score' : [rf_f1,   xgb_f1,   mlp_f1],
    'ROC-AUC'  : [rf_auc,  xgb_auc,  mlp_auc],
})
results = results.sort_values('Accuracy', ascending=False).reset_index(drop=True)
results['Accuracy'] = results['Accuracy'].map('{:.4f}'.format)
results['F1-Score'] = results['F1-Score'].map('{:.4f}'.format)
results['ROC-AUC']  = results['ROC-AUC'].map('{:.4f}'.format)
print("📊 Perbandingan Performa Model:")
print(results.to_string(index=False))


### 7.2 Confusion Matrix (ketiga model)

In [ ]:
class_names = [label_map[i] for i in sorted(label_map)]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Confusion Matrix — Perbandingan Model', fontsize=13, fontweight='bold')

for ax, (pred, title, color) in zip(axes, [
    (rf_pred,  'Random Forest', 'Blues'),
    (xgb_pred, 'XGBoost',       'Greens'),
    (mlp_pred, 'MLP',           'Purples')
]):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap=color)
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', labelrotation=15)

plt.tight_layout()
plt.show()


### 7.3 ROC Curve (One-vs-Rest, per kelas)

In [ ]:
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
class_names_short = ['Low', 'Medium', 'High']
line_styles = ['-', '--', ':']
model_colors = {'Random Forest': '#3266ad', 'XGBoost': '#2ecc71', 'MLP': '#9b59b6'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
fig.suptitle('ROC Curve per Kelas (One-vs-Rest)', fontsize=13, fontweight='bold')

for ax, (model_name, proba) in zip(axes, [
    ('Random Forest', rf_proba),
    ('XGBoost',       xgb_proba),
    ('MLP',           mlp_proba)
]):
    color = model_colors[model_name]
    for i, (cls, ls) in enumerate(zip(class_names_short, line_styles)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
        auc_val = roc_auc_score(y_test_bin[:, i], proba[:, i])
        ax.plot(fpr, tpr, linestyle=ls, color=color,
                label=f'{cls} (AUC={auc_val:.3f})', linewidth=1.8)
    ax.plot([0,1],[0,1],'k--',linewidth=0.8,alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(model_name, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


### 7.4 Bar Chart Perbandingan Metrik

In [ ]:
models_list = ['Random Forest', 'XGBoost', 'MLP']
acc_vals = [rf_acc,  xgb_acc,  mlp_acc]
f1_vals  = [rf_f1,   xgb_f1,   mlp_f1]
auc_vals = [rf_auc,  xgb_auc,  mlp_auc]

x = np.arange(len(models_list))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width, acc_vals, width, label='Accuracy', color='#3266ad',  edgecolor='white')
bars2 = ax.bar(x,         f1_vals,  width, label='F1-Score', color='#e08c3a',  edgecolor='white')
bars3 = ax.bar(x + width, auc_vals, width, label='ROC-AUC',  color='#6b74d4',  edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(models_list, fontsize=11)
ax.set_ylim(0.70, 1.02)
ax.set_ylabel('Score')
ax.set_title('Perbandingan Performa Model', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
plt.show()


## 8. Feature Importance

In [ ]:
# ── Random Forest built-in feature importance ─────────────────────────
rf_importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(15)

# ── XGBoost feature importance ────────────────────────────────────────
xgb_importances = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top 15 Feature Importance', fontsize=13, fontweight='bold')

for ax, (importances, title, color) in zip(axes, [
    (rf_importances,  'Random Forest', '#3266ad'),
    (xgb_importances, 'XGBoost',       '#2ecc71')
]):
    ax.barh(importances.index[::-1], importances.values[::-1],
            color=color, edgecolor='white', height=0.7)
    ax.set_xlabel('Importance Score')
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(importances.values[::-1]):
        ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\n🔍 Top 5 Fitur Paling Berpengaruh (Random Forest):")
for feat, imp in rf_importances.head(5).items():
    print(f"   {feat:40s}: {imp:.4f}")


## 9. Simpan Model Terbaik

In [ ]:
# Tentukan model terbaik berdasarkan accuracy
accs = {
    'Random Forest': rf_acc,
    'XGBoost'      : xgb_acc,
    'MLP'          : mlp_acc
}
best_name = max(accs, key=accs.get)
best_model_obj = {
    'Random Forest': rf_model,
    'XGBoost'      : xgb_model,
    'MLP'          : mlp_model
}[best_name]

print(f"🏆 Model terbaik: {best_name} (Accuracy: {accs[best_name]:.4f})")

# Simpan model, scaler, dan kolom fitur
joblib.dump(best_model_obj, 'best_model.pkl')
joblib.dump(scaler,         'scaler.pkl')
joblib.dump(list(X.columns),'feature_columns.pkl')

print("\n✅ File tersimpan:")
print("   best_model.pkl       → model terbaik")
print("   scaler.pkl           → StandardScaler")
print("   feature_columns.pkl  → daftar nama fitur")


In [ ]:
# ── Download file model ke lokal ──────────────────────────────────────
from google.colab import files

files.download('best_model.pkl')
files.download('scaler.pkl')
files.download('feature_columns.pkl')

print("⬇️  File model siap diunduh.")


---

## Ringkasan

| Aspek | Detail |
|---|---|
| **Dataset** | Enhanced Anxiety Dataset — 11.000 baris, 19 fitur |
| **Task** | Multi-class classification (Low / Medium / High anxiety) |
| **Preprocessing** | Label encoding, One-Hot Encoding, StandardScaler, SMOTE |
| **Model** | Random Forest · XGBoost · MLP |
| **Evaluasi** | Accuracy, F1-Score, ROC-AUC, Confusion Matrix, ROC Curve |
| **Output** | `best_model.pkl`, `scaler.pkl`, `feature_columns.pkl` |

Model terbaik yang tersimpan siap digunakan sebagai backend **Streamlit dashboard**.
